# Phase 2 — Data Quality

Raw Data
   ↓
1. Missing values
   ↓
2. Duplicates
   ↓
3. Invalid values
   ↓
4. Wrong data types
   ↓
5. Inconsistent values
   ↓
6. Decide treatment
   ↓
7. Create cleaned datasets
   ↓
8. Validate cleaned datasets
   ↓
9. Commit to GitHub

 Our cleaned data will eventually go into: data/processed/

 If we make a mistake while cleaning, we can always go back to the original.

In [3]:
import pandas as pd

In [4]:
orders = pd.read_csv("../data/raw/olist_orders_dataset.csv")

order_items = pd.read_csv("../data/raw/olist_order_items_dataset.csv")

products = pd.read_csv("../data/raw/olist_products_dataset.csv")

customers = pd.read_csv("../data/raw/olist_customers_dataset.csv")

sellers = pd.read_csv("../data/raw/olist_sellers_dataset.csv")

payments = pd.read_csv("../data/raw/olist_order_payments_dataset.csv")

reviews = pd.read_csv("../data/raw/olist_order_reviews_dataset.csv")

geolocation = pd.read_csv("../data/raw/olist_geolocation_dataset.csv")

category_translation = pd.read_csv(
    "../data/raw/product_category_name_translation.csv"
)

In [5]:
def missing_summary(df):
    missing=df.isna().sum()
    percentage=(missing/len(df)*100).round(2)

    return pd.DataFrame({
        "missing_count":missing,
        "missing_percentage":percentage
    }).sort_values(
        "missing_percentage",
        ascending=False
    )

## For orders

In [6]:
missing_summary(orders)

,missing_count,missing_percentage
order_delivered_customer_date,2965,2.98
order_delivered_carrier_date,1783,1.79
order_approved_at,160,0.16
order_id,0,0.00
order_purchase_timestamp,0,0.00
order_status,0,0.00
customer_id,0,0.00
order_estimated_delivery_date,0,0.00


In [7]:
#This tells us: Among orders with no customer-delivery date, what is their status


# ex: no delivery date and cancelled=619
#OR ex: 619 cancelled orders have no delivery date

# ex: no delivery date and shipped 1107
#OR ex: 1,107 shipped orders have no delivery date
orders[
    orders["order_delivered_customer_date"].isna()
]["order_status"].value_counts()

order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64

In [8]:
orders["order_status"].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [9]:
orders[
    orders["order_approved_at"].isna()
]["order_status"].value_counts()

order_status
canceled     141
delivered     14
created        5
Name: count, dtype: int64

In [10]:
orders[
    orders["order_delivered_carrier_date"].isna()
]["order_status"].value_counts()

order_status
unavailable    609
canceled       550
invoiced       314
processing     301
created          5
approved         2
delivered        2
Name: count, dtype: int64

In [ ]:

#The first checks duplicate order IDs.
orders["order_id"].duplicated().sum()

np.int64(0)

In [13]:
#The second checks whether the entire row is duplicated.
orders.duplicated().sum()

np.int64(0)

In [14]:
order_items.isna().sum()

order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64

In [ ]:
#Why sort it?
#Because we want the columns with the most missing values at the top.
order_items.isna().sum().sort_values(ascending=False)

order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64

In [18]:
(order_items["price"]<0).sum()

np.int64(0)

In [19]:
(order_items["freight_value"]<0).sum()

np.int64(0)

In [21]:
# "Is the value missing?"
(order_items["price"].isnull()).sum()

#"Is the value zero?"
(order_items["price"]==0).sum()


np.int64(0)

Check product dimensions

In [23]:
products.info()

<class 'pandas.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  32951 non-null  str    
 1   product_category_name       32341 non-null  str    
 2   product_name_lenght         32341 non-null  float64
 3   product_description_lenght  32341 non-null  float64
 4   product_photos_qty          32341 non-null  float64
 5   product_weight_g            32949 non-null  float64
 6   product_length_cm           32949 non-null  float64
 7   product_height_cm           32949 non-null  float64
 8   product_width_cm            32949 non-null  float64
dtypes: float64(7), str(2)
memory usage: 2.3 MB


In [24]:
(products["product_weight_g"]<0).sum()

np.int64(0)

In [29]:
#Single bracket → one column
#Double bracket → multiple columns
#The inner [] is a Python list containing the column names:
#The outer [] tells Pandas:"Select these columns from the DataFrame."

(products[["product_height_cm","product_length_cm","product_width_cm"]]<0).sum()

product_height_cm    0
product_length_cm    0
product_width_cm     0
dtype: int64

Check reviews


In [32]:
reviews["review_score"].value_counts().sort_index()

review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64

The review score should be within the expected rating range.

In [30]:
reviews["review_score"].min()

np.int64(1)

In [31]:
reviews["review_score"].max()

np.int64(5)

Step 3: Fix data types

In [ ]:
#Identify the date columns

date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

In [ ]:
for col in date_columns:
    orders[col]=pd.to_datetime(
        orders[col],
        errors="coerce"#If Pandas cannot convert a value properly, instead of giving an error, convert that value to NaN
        )